# Task 2
## Integrantes
- Sergio Orellana 221122
- Rodrigo Mansilla 22611
- Ricardo Chuy 221007

En el pabellón principal del evento, se está llevando a cabo un concurso de cosplay. De repente, un grupo de 15 personas haciendo cosplay de Naruto entran al escenario exactamente iguales y posan muy juntos (simulando el Kage Bunshin no Jutsu o Jutsu Clones de Sombra). Las cámaras registran a los clones abrazados, superpuestos y parcialmente ocluidos unos detrás de otros. 

Usted nota que su modelo clásico basado en YOLOv8 falla rotundamente: la red neuronal sí encuentra a los 15 clones y dibuja cientos de cajas, pero al final en la pantalla solo aparecen 4 o 5 personas detectadas. El resto "desaparece". Considerando esto, responda:

1. **Explique matemáticamente, utilizando la fórmula del IoU (Intersección sobre Unión), por qué el algoritmo NMS está causando que los clones superpuestos desaparezcan (Falsos Negativos).**

El detector genera muchas cajas candidatas para los 15 clones, y luego el NMS intenta quedarse solo con una caja “ganadora” cuando detecta que varias cajas se superponen mucho. Si tengo dos cajas $B_i$ y $B_j$ con scores $s_i > s_j$, y además $IoU(B_i,B_j) = area_interseccion / area_union > theta_NMS$, entonces el NMS conserva $B_i$ y elimina $B_j$. Ese es justamente el mecanismo básico del NMS, ordenar por confianza y suprimir cajas con alta superposición respecto a una caja de mayor score.

El problema es que, en una escena como esta, varias cajas correctas pueden pertenecer a personas distintas y aun así tener un IoU alto, porque los clones están abrazados, uno detrás de otro y parcialmente ocluidos. Entonces, el NMS comete una confusión estructural: interpreta “dos personas muy juntas” como si fueran “muchas cajas repetidas del mismo objeto”. En consecuencia, cajas válidas de clones reales quedan suprimidas y eso se convierte en falsos negativos. Es decir, la red sí ve a los 15, pero el postprocesamiento borra parte de esas detecciones.

2. **Si usted es el ingeniero a cargo y solo puede modificar los hiperparámetros en el código durante la inferencia: ¿Qué pasaría si ajusta el umbral de IoU del NMS a 0.15? ¿Qué pasaría si lo ajusta a 0.95? Justifique qué valor sería más adecuado para este problema de alta densidad.**

Si yo ajusto el umbral a $0.15$, el NMS se vuelve extremadamente agresivo. Basta con que dos cajas se solapen un poco para que una de ellas sea eliminada. En una multitud tan densa, eso empeora el problema: muchas cajas de personas distintas superarían ese umbral y desaparecerían todavía más clones; eeduciría duplicados, sí, pero a costa de destruir muchas detecciones verdaderas.

Si lo ajusto a $0.95$, ocurre lo contrario: el NMS casi no suprime nada, porque solo eliminaría cajas casi idénticas. Eso ayudaría a conservar más clones reales, pero también dejaría muchísimas cajas duplicadas alrededor de cada persona. Entonces mejoraría el recall, pero la salida sería muy ruidosa y la precisión visual del sistema caería, porque vería varias cajas para el mismo clon.

Para este problema de alta densidad, yo elegiría un valor alto, pero no extremo. No usaría $0.15$, y tampoco me iría hasta $0.95$. Mi decisión sería moverlo hacia una zona más permisiva, por ejemplo alrededor de $0.75$ a $0.85$, porque necesito tolerar superposición entre personas distintas sin dejar que el modelo inunde la escena con duplicados casi idénticos. En este caso, me conviene relajar el NMS, no endurecerlo.

3. **Si el presupuesto le permitiera cambiar el modelo a YOLOv10, explique cómo su arquitectura Asignación Dual de Etiquetas (Dual Label Assignment) resolvería este problema de oclusión sin necesidad de tocar el NMS.**

Yo esperaría una mejora importante porque su propuesta central introduce consistent dual assignments para entrenamiento sin NMS. La idea de fondo es que el modelo deje de depender tanto del esquema clásico de “generar muchas cajas y luego limpiarlas con una supresión codiciosa”. En lugar de eso, YOLOv10 está diseñado para entrenamiento NMS-free, lo que reduce la dependencia de ese postprocesamiento y también baja la latencia de inferencia.

Desde la perspectiva del problema de oclusión, eso me ayuda porque el fallo ya no queda dominado por una regla codiciosa que suprime cajas solo por tener alto solapamiento. Pero en este caso (personas abrazadas), el modelo puede aprender una asignación más consistente entre predicciones y objetos reales, en lugar de castigar automáticamente a cajas vecinas por parecerse demasiado entre sí. Por lo que, YOLOv10 me ayudaría a reducir el riesgo de que una persona desaparezca solo porque otra, muy cercana, obtuvo un score ligeramente mayor.




# Referencias

- GeeksforGeeks. (2025a, April 26). What is NonMaximum Suppression? GeeksforGeeks. https://www.geeksforgeeks.org/what-is-non-maximum-suppression/ 
- GeeksforGeeks. (2025b, July 23). Evaluating Object Detection Models: Methods and Metrics. GeeksforGeeks. https://www.geeksforgeeks.org/evaluating-object-detection-models-methods-and-metrics/ 
- GeeksforGeeks. (2025c, July 23). Mean Average precision (MAP) in computer vision. GeeksforGeeks. https://www.geeksforgeeks.org/mean-average-precision-map-in-computer-vision/ 
- GeeksforGeeks. (2025d, July 23). Object detection models. GeeksforGeeks. https://www.geeksforgeeks.org/object-detection-models/ 
- GeeksforGeeks. (2025e, July 23). Object Detection using yolov8. GeeksforGeeks. https://www.geeksforgeeks.org/object-detection-using-yolov8/